# Lumen Clip Kaggle GPU backend

Enable Settings -> Accelerator -> GPU first.
Model: ali-vilab/text-to-video-ms-1.7b

## 1. Install dependencies

In [ ]:
import subprocess, sys
pkgs = ['diffusers>=0.29.0','transformers>=4.41.0','accelerate>=0.31.0','safetensors>=0.4.3','fastapi>=0.111.0','uvicorn>=0.30.0','imageio>=2.34.0','imageio-ffmpeg>=0.5.1','opencv-python-headless>=4.10.0','pydantic>=2.7.0']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
print('deps ready')

## 2. Check GPU

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name', torch.cuda.get_device_name(0))
    print('vram_gb', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise SystemExit('Enable GPU in Kaggle Settings then Restart session.')

## 3. Load project files

In [ ]:
from pathlib import Path
import urllib.request, sys
REPO = 'https://raw.githubusercontent.com/sgue19000/t2v-kaggle-webapp/main/kaggle'
workdir = Path('/kaggle/working')
for name in ('generator.py', 'server.py'):
    urllib.request.urlretrieve(f'{REPO}/{name}', workdir / name)
    print('wrote', name)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

## 4. Download / load model

In [ ]:
from generator import load_pipeline, model_info
load_pipeline()
print(model_info())

## 5. Define video generation function

In [ ]:
from pathlib import Path
from generator import generate_video
def make_clip(prompt: str):
    out = Path('/kaggle/working/outputs/demo.mp4')
    return generate_video({'prompt': prompt, 'num_frames': 16, 'height': 256, 'width': 256, 'fps': 8, 'guidance_scale': 9, 'seed': 42, 'num_inference_steps': 20}, out_path=out)

## 6. Start API server

In [ ]:
import os, time, threading, subprocess, re
from pathlib import Path
os.environ['T2V_OUTPUT_DIR'] = '/kaggle/working/outputs'
Path(os.environ['T2V_OUTPUT_DIR']).mkdir(parents=True, exist_ok=True)
def run_api():
    import uvicorn
    from server import app
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')
threading.Thread(target=run_api, daemon=True).start()
time.sleep(3)
print('API listening on :8000')
cf = Path('/kaggle/working/cloudflared')
if not cf.exists():
    subprocess.check_call(['wget', '-q', '-O', str(cf), 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
    cf.chmod(0o755)
log_path = Path('/kaggle/working/tunnel.log')
log = open(log_path, 'w')
subprocess.Popen([str(cf), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'], stdout=log, stderr=log)
url = None
for _ in range(40):
    time.sleep(1)
    text = log_path.read_text(errors='ignore')
    found = re.findall(r'https://[-a-z0-9.]+trycloudflare.com', text)
    if found:
        url = found[-1]
        break
print('PUBLIC_API_URL=', url or 'check /kaggle/working/tunnel.log')

## 7. Test generation

In [ ]:
from pathlib import Path
demo = make_clip('A golden retriever running through tall grass at sunrise')
print(demo, 'bytes', Path(demo).stat().st_size)

## 8. Display generated video

In [ ]:
from IPython.display import Video, display
from pathlib import Path
p = Path('/kaggle/working/outputs/demo.mp4')
display(Video(str(p), embed=True)) if p.exists() else print('Run the test cell first.')